# 🚀 Huấn Luyện Agent 3: Academic Summarizer (Generator SFT + DPO + Critic) Trên Google Colab T4
### Fine-tuning mô hình Tóm tắt học thuật đa tầng với kiến trúc LoRA / QLoRA
- **Phần cứng đề xuất**: Google Colab T4 GPU (16GB VRAM)
- **Tối ưu**: LoRA FP16 / 4-bit QLoRA, Batch size = 1, Max length = 1536


In [ ]:
def patch_environment():
    """Patches missing triton.ops, language attributes, and handles CPU/GPU fallbacks for Kaggle and Colab."""
    import os
    import sys
    import types
    import importlib.machinery

    if "BNB_CUDA_VERSION" in os.environ:
        del os.environ["BNB_CUDA_VERSION"]

    # 1. Patch Triton mock safely
    m_triton = sys.modules.get("triton")
    if m_triton is None:
        try:
            import triton
            m_triton = triton
        except Exception:
            m_triton = types.ModuleType("triton")
            m_triton.__spec__ = importlib.machinery.ModuleSpec("triton", None)
            m_triton.__path__ = []
            sys.modules["triton"] = m_triton

    # Ensure triton.language exists to satisfy transformers.modeling_utils import checks
    if not hasattr(m_triton, "language"):
        m_lang = types.ModuleType("triton.language")
        m_lang.__spec__ = importlib.machinery.ModuleSpec("triton.language", None)
        m_lang.constexpr = lambda x: x
        setattr(m_triton, "language", m_lang)
        sys.modules["triton.language"] = m_lang

    if not hasattr(m_triton, "cdiv"):
        setattr(m_triton, "cdiv", lambda a, b: (a + b - 1) // b)
    if not hasattr(m_triton, "jit"):
        setattr(m_triton, "jit", lambda *a, **k: (lambda f: f))

    m_ops = types.ModuleType("triton.ops")
    m_ops.__spec__ = importlib.machinery.ModuleSpec("triton.ops", None)
    m_ops.__path__ = []
    m_perf = types.ModuleType("triton.ops.matmul_perf_model")
    m_perf.__spec__ = importlib.machinery.ModuleSpec("triton.ops.matmul_perf_model", None)
    m_perf.early_config_prune = lambda *a, **k: None
    m_perf.estimate_matmul_time = lambda *a, **k: 0
    m_ops.matmul_perf_model = m_perf
    setattr(m_triton, "ops", m_ops)
    sys.modules["triton.ops"] = m_ops
    sys.modules["triton.ops.matmul_perf_model"] = m_perf

    # 2. Bulletproof fallback for bitsandbytes
    try:
        import bitsandbytes as bnb
    except Exception:
        import torch
        m_bnb = types.ModuleType("bitsandbytes")
        m_bnb.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes", None)
        m_bnb_nn = types.ModuleType("bitsandbytes.nn")
        m_bnb_nn.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes.nn", None)

        class DummyBnbLinear(torch.nn.Linear):
            pass

        m_bnb_nn.Linear8bitLt = DummyBnbLinear
        m_bnb_nn.Linear4bit = DummyBnbLinear
        m_bnb_nn.modules = types.ModuleType("bitsandbytes.nn.modules")
        m_bnb_nn.modules.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes.nn.modules", None)
        m_bnb.nn = m_bnb_nn
        m_bnb.__version__ = "0.47.0"
        sys.modules["bitsandbytes"] = m_bnb
        sys.modules["bitsandbytes.nn"] = m_bnb_nn
        sys.modules["bitsandbytes.nn.modules"] = m_bnb_nn.modules

patch_environment()

# Kiểm tra phần cứng và thư viện
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  - GPU {i}: {torch.cuda.get_device_name(i)}')
else:
    print('⚠️ CẢNH BÁO: BẠN CHƯA BẬT GPU!')
    print('👉 Vui lòng vào menu Runtime -> Change runtime type -> Chọn T4 GPU (hoặc Dual T4 trên Kaggle) -> Nhấn Save và chạy lại từ đầu.')


In [ ]:
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN ĐÃ ĐƯỢC CỐ ĐỊNH PHIÊN BẢN (PINNED SOTA STACK)
!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    trl==0.12.1 \
    sentence-transformers==3.2.1 \
    bitsandbytes \
    accelerate>=1.1.0 \
    datasets==3.0.2 \
    safetensors==0.4.5 \
    ninja PyMuPDF scikit-learn scipy einops ipywidgets

print('✅ Đã cài đặt xong toàn bộ thư viện tương thích với Tesla T4!')


In [ ]:
# BƯỚC 2: CLONE MÃ NGUỒN VÀ CHUẨN BỊ THƯ MỤC
import os, sys, shutil

!rm -rf /content/Research-assist
!git clone https://github.com/nguyentatmanh/Research-assist.git /content/Research-assist
%cd /content/Research-assist
!git checkout upgrade 2>/dev/null || true

import torch
if torch.cuda.is_available():
    print(f"✅ GPU Đang Dùng: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()


In [ ]:
# ==============================================================================
# BƯỚC 3: KHỞI CHẠY HUẤN LUYỆN AGENT 3 (GENERATOR SFT + DPO + CRITIC)
# ==============================================================================
import os, sys, torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Chạy với tham số tối ưu VRAM: batch=1, max_len=1536
!python training_lab/agent3_summarizer/train_agent3_t4x2.py --role both --epochs-sft 3 --epochs-dpo 2 --batch-size 1 --max-length 1536


In [ ]:
# ==============================================================================
# BƯỚC 4: DỌN DẸP & ĐÓNG GÓI TRỌNG SỐ TẢI VỀ MÁY TÍNH
# ==============================================================================
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

weights_dir = Path("training_lab/agent3_summarizer/weights")
zip_path = "Agent3_Academic_Summarizer.zip"

if not weights_dir.exists():
    print("⚠️ Chưa tìm thấy thư mục weights của Agent 3. Vui lòng đảm bảo bước huấn luyện hoàn tất thành công.")
else:
    print("⏳ Đang dọn dẹp và tối ưu dung lượng...")
    !rm -rf training_lab/agent3_summarizer/weights/**/checkpoints*
    !zip -r -q {zip_path} training_lab/agent3_summarizer/weights/

    print(f"\n🎉 ĐÃ ĐÓNG GÓI THÀNH CÔNG: {zip_path}")
    print("👉 Bấm vào nút màu xanh bên dưới để tải về máy tính:")

    download_btn = widgets.Button(
        description="📥 Click để tải Agent3_Academic_Summarizer.zip",
        button_style="success",
        layout=widgets.Layout(width="440px", height="50px")
    )

    def on_click_download(b):
        if os.path.exists(zip_path):
            print("🚀 Trình duyệt đang bắt đầu tải Agent 3 về máy tính...")
            files.download(zip_path)
        else:
            print(f"❌ Không tìm thấy file {zip_path}.")

    download_btn.on_click(on_click_download)
    display(download_btn)
